# 032-optimised__ks_mu_sigma — GM record selection, AvgSA([0, 3]), KS + moments selection `__ks_mu_sigma` — Stage 1 (compute)

The fourth **complete** record selection for every `(site, iml)` stripe, to be compared with
the original (nb `032`) and the KS-optimised (nb `032-…-optimised`) selections. Same engine
(`build_final_ensembles`), same GCIM targets, same database filter; what differs is **the
objective** the selection minimises and **the IM weights**.

**Why.** The KS-optimised selection improved the KS fit but pushed the set *moments* of ln IM away
from the GCIM target, most of all for Ds5-95 (set SD a median 1.44 × σ_GCIM, nb 034): KS is nearly
blind to a few extreme records, each of which moves the ECDF by only 1/30. The set moments are used
as regression covariates, so this selection targets them directly.

Per stripe and selection IM $m$ (conditioning IM dropped), over the 30 selected records:

$$z_{\mu,m} = \frac{\overline{\ln IM}_m - \mu_{GCIM,m}}{\sigma_{GCIM,m}},\qquad
r_{\sigma,m} = \frac{s_{\ln IM,m}}{\sigma_{GCIM,m}}$$

with the set mean and sample SD (`ddof=1`) of ln IM (`ims_scaled`), and the GCIM target mean and
SD from nb 031's `gcim_dists[(site, iml)]["stats"]`, looked up by IM name.

$$J' = c\,V + \tilde\lambda_{KS}\sum_m w_m D_m + \tilde\lambda_\mu\sum_m w_m z_{\mu,m}^2
+ \tilde\lambda_\sigma\sum_m w_m(\ln r_{\sigma,m})^2,\qquad
\tilde\lambda_k = \frac{\lambda_k}{\lambda_{KS}+\lambda_\mu+\lambda_\sigma} = \tfrac13$$

$c\,V$ ($c = 10$) is the KS-band violation penalty of the KS-optimised selection, kept
**outside** the normalisation: it is the constraint that keeps the set inside the KS band. The
selection **is** still required to pass KS (acceptance `"ks"`, as before). Equal λ's are not
equal influence: the three metrics live on different natural scales.

**IM weights**: Ds5-95 **0.40**, each of the 21 spectral IMs 0.60/21 (`weight_rsd595` in the scheme
config), throughout: SSE record picking, best-of-20 candidate choice, optimiser, keep-better.

| round | select with | optimise with | which stripes are optimised |
|---|---|---|---|
| 1 | bounds on **M and Vs30, not distance** | same | **every** stripe (`force_optimisation=True`) |
| 2 | **no bounds**; only stripes for which round 1 could not form a record set at all are reselected | no bounds | every stripe still failing KS |
| 3 | no reselection | no bounds, best of 5 shuffled database orderings | stripes still failing KS; keeps the better of {new, old} scored with $J'$ |

The same rounds as the KS-optimised selection; every scoring step uses $J'$.

The scheme (rounds, objective, its parameters, acceptance, weight) is `SELECTION_CONFIG_KS_MU_SIGMA` in
`setup_AvgSA03_gm_selection.py`, imported **once** below as `SCHEME`. The selection context is
built with `setup_AvgSA03_gcim_gm_selection_for_scheme(SCHEME, ...)`, which puts the scheme's
weight and objective into it; the engine reads them from there at every scoring step, and the
scheme enters every stripe / stage / final fingerprint. Changing a value in the scheme config
therefore reselects this selection's stripes, and no other selection's.

**Excluded records.** The database is filtered exactly as in the other selections, once, in the
cell "Set up + filter the database" (`gm_db_for_selection`, all versions of
`UNAVAILABLE_RECORDS`); the exclusion is recorded in every manifest (`db_exclusions`).

**Kept apart from the other selections.** Everything this notebook writes goes to
`cfg["proc_data"]["gm_selection_optimised__ks_mu_sigma"]` (`data_processed/06_gm_selection/AvgSA_03_optimised__ks_mu_sigma/`)
and `cfg["results"]["AvgSA_03_record_selection_optimised__ks_mu_sigma"]`
(`results/04_record_selection/AvgSA_03_optimised__ks_mu_sigma/`), under the same file names as the original.
The original and KS-optimised selections' files are never read or written here.

**Incremental**, exactly like nb `032`: `find_stale_stripes` (against this selection's result
folder and `SCHEME`) picks the stripes that are missing or stale, and the batch is merged into
`AvgSA_03_final_ensembles.pickle` in this selection's folder, which stays the COMPLETE
`(site, iml)` set. On the first run every stripe is to (re)select. Set `FORCE_RECOMPUTE = True`
to reselect every wanted stripe.

**Upstream**:

| Input | Source |
|---|---|
| GCIM target distributions (complete set) | `gcim_dist_AvgSA_03.pickle` (nb `031`) — **shared**, read only |
| Combined selection GM database | `cfg["proc_data"]["gm_database"]`, filtered in memory (see above) |
| IML-based disagg shards + poe stats | `cfg["proc_data"]["AvgSA_03_disagg_data_shards"]`, `cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| GMM logic tree + correlation flatfiles | loaded inside `setup_AvgSA03_gcim_gm_selection_for_scheme()` |

**Downstream** — `033-gm_selection_AvgSA_03_stage2_postprocess-optimised__ks_mu_sigma.ipynb`.

**Run order** — run top to bottom, then `033-…-optimised__ks_mu_sigma`. The first run selects all
stripes (over an hour; every stripe is optimised in round 1).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    build_final_ensembles,
    find_stale_stripes,
    ensemble_accepted,
    normalise_selection_scheme,
)
# NOTE: SELECTION_CONFIG_KS_MU_SIGMA (imported once, used below as SCHEME), NOT another
# selection's config. Mixing them up would fingerprint the stripes against the wrong
# scheme (stripe_fingerprints refuses a context built for another scheme).
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection_for_scheme,
    wanted_stripe_keys,
    SELECTION_CONFIG_KS_MU_SIGMA,
    stripe_source_fps,
    stripe_fingerprints,
    gm_db_for_selection,
)


cfg = load_config()

# THE selection scheme of this notebook: rounds, objective + parameters, acceptance,
# Ds5-95 weight. Every use below reads it from here; nothing is typed in the notebook.
SCHEME = SELECTION_CONFIG_KS_MU_SIGMA

In [ ]:
# All OUTPUTS of this notebook live in the optimised folder; the original selection's
# folder (cfg["proc_data"]["gm_selection"]) is never written to.
GMS = cfg["proc_data"]["gm_selection_optimised__ks_mu_sigma"]
GMS.mkdir(parents=True, exist_ok=True)

# canonical Stage-1 output (merged each run to stay the COMPLETE (site, iml) set).
# Same file name as the original, in the optimised folder, so nb 040-optimised__ks_mu_sigma's
# summarise_availability() finds it unchanged.
final_ensembles_fp = GMS / "AvgSA_03_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded; per-batch scratch).
# Named {im}_rd{ii}_{selection|optimisation}.pickle, in the optimised folder.
IM = "AvgSA_03"
n_rounds = len(SCHEME["round_unbounded"])
stage_fps = {
    "select":   [GMS / f"{IM}_rd{ii}_selection.pickle"    for ii in range(1, n_rounds + 1)],
    "optimise": [GMS / f"{IM}_rd{ii}_optimisation.pickle" for ii in range(1, n_rounds + 1)],
}

# Inputs used for provenance fingerprinting (hashed by their file bytes). These stage /
# final artifacts cover a whole batch, so the disagg entry is the digest of the ENTIRE
# shard set (disagg_shard_dir -> disagg_shards_digest), not any one site's shard — see
# gm_selection._disagg_fingerprint_input. The GCIM file is the ORIGINAL nb 031 output
# (read only): the targets do not depend on the bounds.
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_shard_dir":  cfg["proc_data"]["AvgSA_03_disagg_data_shards"],
    "disagg_stats_file": cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"],
    "site_model_file":   cfg["hazard_models"]["eshm20_wp1_site_model"],
}

# The selection scheme (rounds, bounds, shuffle, seeds, rng_seed, db_exclusions, objective,
# acceptance, weight_rsd595) lives in SELECTION_CONFIG_KS_MU_SIGMA (setup_AvgSA03_gm_selection.py),
# so nbs 032/033-optimised__ks_mu_sigma fingerprint each stripe identically. See that constant for
# the round-by-round description.
rng_seed = SCHEME["rng_seed"]

# Escape hatch: force-reselect ALL wanted stripes, ignoring the per-stripe manifests
# (and bypassing every stage's staleness guard). Leave False for normal incremental
# runs: only the stale/new (site, iml) are (re)selected.
FORCE_RECOMPUTE = False

In [ ]:
# --- Set up + filter the database ---------------------------------------------------
# Set up the selection context WITHOUT loading any disagg data (sites=()) — the stale
# check needs only the context and the index-derived key set. The shards for the stale
# sites are loaded in the compute cell, once we know which sites those are.
# _for_scheme(SCHEME, ...): the context carries the scheme's Ds5-95 weight and objective,
# which the engine reads at every scoring step and the fingerprints include.
_, disagg_stats, site_model, basic_selection_ctx, gm_db_full = setup_AvgSA03_gcim_gm_selection_for_scheme(SCHEME, sites=())

# THE database filter (the only place it happens; shared with nb 032): drop every
# record in UNAVAILABLE_RECORDS -- all versions of the list -- in memory, keeping the
# original index labels. From here on ONLY `gm_db` (filtered) is used; `gm_db_full` is
# kept for the check at the end of the notebook.
gm_db = gm_db_for_selection(gm_db_full)
excluded_idx = gm_db_full.index.difference(gm_db.index)
if len(excluded_idx) == 0:
    raise RuntimeError("The filter dropped no rows: UNAVAILABLE_RECORDS matches nothing in "
                       "the database, so the exclusion is not in effect.")

# Incremental cache: (re)select only stripes missing or stale in the OPTIMISED result
# folder. The fingerprint uses SCHEME, which carries db_exclusions,
# so the exclusion is part of every stripe's manifest.
#
# The check is versioned by exclusion list (stripe_fingerprints): a stripe selected under
# an older version of UNAVAILABLE_RECORDS stays valid as long as it contains none of the
# records excluded since. Only a stripe that contains a newly excluded record is
# reselected.
RESULT_FOLDER = cfg["results"]["AvgSA_03_record_selection_optimised__ks_mu_sigma"]
RESULT_FOLDER.mkdir(parents=True, exist_ok=True)
source_fps_stripe = stripe_source_fps()
wanted = wanted_stripe_keys()
fp_fn, superseded = stripe_fingerprints(
    SCHEME, source_fps_stripe, basic_selection_ctx, disagg_stats)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn, superseded=superseded)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)select.")

# load the (complete) gcim distributions written by nb 031 (shared with the original)
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print(f"Loaded gcim for {len(gcim_dists)} (site, iml).")
else:
    gcim_dists = {}
    print("No GCIM distribution data found — run nb 031 first.")

## Stage 1 - Build final ensembles (slow compute)

Runs the multi-round engine with `SELECTION_CONFIG_KS_MU_SIGMA` (see the table at the top) on the
**filtered** database and saves `AvgSA_03_final_ensembles.pickle` (+ a `.manifest.json`) in the
optimised folder. `db_exclusions` is passed to the engine so the stage caches and the final
manifest record the filter. Each step is provenance-cached: if an input changed since a cache
was written, the stage is recomputed; set `FORCE_RECOMPUTE = True` to rebuild.

Check in the printout: round 1 should read `Bounds  [bounded: m, vs30 | unbounded: d]` and
its optimisation phase should cover **all** stripes in the batch (`[force: ...]`).

Post-processing lives in **`033-gm_selection_AvgSA_03_stage2_postprocess-optimised__ks_mu_sigma.ipynb`**.

In [ ]:
# Force override: reselect every wanted stripe (ignores the per-stripe manifests).
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

# Capture the prior COMPLETE final_ensembles before build_final_ensembles overwrites
# the file with just this batch. On a forced full rebuild we start clean.
prior_final = {}
if final_ensembles_fp.is_file() and not FORCE_RECOMPUTE:
    with open(final_ensembles_fp, "rb") as f:
        prior_final = pickle.load(f)

# Subset to the stale batch and run the multi-round engine over just those keys. The
# round loop is per-(site, iml) independent, so selecting a subset is equivalent to
# selecting them within the full set. gcim for every batch key must be present (built
# by nb 031); a missing key means 031 was not run for this batch.
#
# The disagg shards are loaded HERE, and only for the sites with stale stripes — an
# all-valid run reads no disagg data at all. The `gm_db` this call returns is the
# UNFILTERED database and is deliberately discarded (`*_`): only the filtered `gm_db`
# from the set-up cell is ever passed on.
if to_compute:
    stale_sites = {s for s, _ in to_compute}
    site_iml_disaggs, *_ = setup_AvgSA03_gcim_gm_selection_for_scheme(SCHEME, sites=stale_sites)
    batch = {k: site_iml_disaggs[k] for k in to_compute}
else:
    batch = {}

missing = [k for k in batch if k not in gcim_dists]
if missing:
    raise KeyError(
        f"{len(missing)} stale (site, iml) have no gcim — run nb 031 first. "
        f"e.g. {missing[:5]}")

if batch:
    batch_final = build_final_ensembles(
        batch,
        disagg_stats,
        gcim_dists,
        gm_db,                      # FILTERED database (set-up cell)
        basic_selection_ctx,
        site_model,
        source_fps=source_fps,
        stage_fps=stage_fps,
        output_fp=final_ensembles_fp,
        round_unbounded=SCHEME["round_unbounded"],
        force_optimisation=SCHEME["force_optimisation"],
        shuffle=SCHEME["shuffle"],
        n_shuffles=SCHEME["n_shuffles"],
        shuffle_rng_seeds=SCHEME["shuffle_rng_seeds"],
        rng_seed=rng_seed,
        force_recompute=FORCE_RECOMPUTE,
        # records the in-memory filter in the stage + final manifests (does not filter)
        db_exclusions=SCHEME["db_exclusions"],
    )
else:
    batch_final = {}
    print("All stripes valid — nothing to (re)select.")

# MERGE the batch into the prior complete set so final_ensembles.pickle stays the
# COMPLETE (site, iml) selection (read as complete by nb 040-optimised__ks_mu_sigma). Prune any key
# no longer wanted (an IML removed from a union list).
wanted_set = set(wanted)
final_ensembles = {k: v for k, v in {**prior_final, **batch_final}.items() if k in wanted_set}
if batch_final or prior_final:
    with open(final_ensembles_fp, "wb") as f:
        pickle.dump(final_ensembles, f)
print(f"final_ensembles.pickle now holds {len(final_ensembles)} (site, iml) "
      f"({len(batch_final)} (re)selected this run).")

In [ ]:
# Stripes WITHOUT AN ACCEPTED record set after all rounds (across the full merged set).
# "Accepted" is the scheme's acceptance criterion, read from SCHEME through the same helper the
# engine and nb 033 use (gm_selection.ensemble_accepted):
#   "complete_set" (__mu_sigma)  a full 30-record set exists; KS is NOT required
#   "ks"           (__ks_mu_sigma) the set passes KS, as in the other selections
# KS is reported either way -- as the gate, or as a diagnostic. Not pickled - just a quick look.
acceptance = normalise_selection_scheme(SCHEME)["acceptance"]
n_samples = basic_selection_ctx["n_samples"]

not_accepted = [k for k, v in final_ensembles.items()
                if not ensemble_accepted(v, acceptance, n_samples)]
no_set = [k for k, v in final_ensembles.items() if v is None]
failing_ensembles = {
    k: v["ks_failed_ims"]
    for k, v in final_ensembles.items()
    if v is not None and not v["ks_passed"]
}

print(f"acceptance = {acceptance!r}: {len(not_accepted)} (site, iml) without an accepted set")
for k in not_accepted:
    print("  ", k)

ks_role = "the gate" if acceptance == "ks" else "a diagnostic only"
print(f"\nKS ({ks_role}): {len(failing_ensembles)} of {len(final_ensembles)} (site, iml) fail KS")
# Under "complete_set" this list can be long; show the first 20 (the count is the point).
shown = list(failing_ensembles) if acceptance == "ks" else list(failing_ensembles)[:20]
for k in shown:
    print("  ", k, "->", failing_ensembles[k])
if len(shown) < len(failing_ensembles):
    print(f"   ... and {len(failing_ensembles) - len(shown)} more")

# A stripe with NO set at all would crash summarise_availability in nb 040 (it reads every entry
# of final_ensembles). Stop here, where the cause is visible, instead.
if no_set:
    raise RuntimeError(f"{len(no_set)} (site, iml) have no record set after all rounds: {no_set}. "
                       f"Decide how to handle them before running nb 033/040.")


## Check: no excluded record was selected

Every selected record is identified by its row label in the database. Since the filter keeps
the original labels, an excluded record would show up here as one of the dropped labels.

In [ ]:
# Look for any dropped row label in every ensemble of the (merged, complete) set.
leaks = {
    k: sorted(set(v["recs"].index).intersection(excluded_idx))
    for k, v in final_ensembles.items()
    if v is not None
}
leaks = {k: v for k, v in leaks.items() if v}
if leaks:
    raise RuntimeError(f"Excluded records were selected in {len(leaks)} stripe(s): {leaks}")
print(f"OK - none of the {len(excluded_idx)} excluded database rows appears in any of the "
      f"{sum(v is not None for v in final_ensembles.values())} ensembles.")